# DL & GenAI project milestone - 4 (23f3001252)

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load datasets

In [3]:
## Load datasets
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [4]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Sample Shape:", sample.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)
Sample Shape: (500, 2)


In [5]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Question 1
#### Q1. Label Encoding Convert the answer column in train.csv into numeric labels using the following mapping: A = 0 B = 1 C = 2 D = 3 E = 4 What is the encoded numeric label for the row at index 150?

In [7]:
# Multiple-Choice Data Formatting

import pandas as pd


# Q1. Label Encoding
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["label"] = train_df["answer"].map(label_map)

print("Encoded label at index 150:")
print(train_df.loc[150, "label"])

Encoded label at index 150:
2


## Question 2
#### Q2. Prompt-Option Formatting For row index 0, create the Option B input using exactly this format: str(prompt) + " [SEP] " + str(option_B) What is the exact character length of this formatted input string? *

In [8]:
# Q2. Prompt-Option Formatting
formatted_input = (
    str(train_df.loc[0, "prompt"])
    + " [SEP] "
    + str(train_df.loc[0, "B"])
)

print("\nFormatted Option B Input:\n")
print(formatted_input)

print("\nCharacter Length:")
print(len(formatted_input))


Formatted Option B Input:

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Character Length:
407


## Question - 3
#### Tokenization for Multiple-Choice Models

Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length
Since each question has five options, every row becomes five tokenized sequences.

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [9]:
# Tokenization for Multiple-Choice Models

import pandas as pd
import torch
from transformers import AutoTokenizer


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


# Q3. Single-Row MCQ Tokenization

row = train_df.iloc[0]

choices = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

prompts = [str(row["prompt"])] * 5

encoding = tokenizer(
    prompts,
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for Multiple Choice model
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

print("Input IDs Shape:", input_ids.shape)
print("Attention Mask Shape:", attention_mask.shape)

print("Second Dimension =", input_ids.shape[1])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs Shape: torch.Size([1, 5, 128])
Attention Mask Shape: torch.Size([1, 5, 128])
Second Dimension = 5


## Question - 4
Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.
The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?


In [10]:
# Q4. Batch MCQ Tokenization

all_input_ids = []
all_attention_masks = []

for _, row in train_df.iloc[:16].iterrows():

    prompts = [str(row["prompt"])] * 5

    choices = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    enc = tokenizer(
        prompts,
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(enc["input_ids"])
    all_attention_masks.append(enc["attention_mask"])

input_ids = torch.stack(all_input_ids)
attention_mask = torch.stack(all_attention_masks)

print("\nBatch Input IDs Shape:", input_ids.shape)

# Total token positions
total_positions = input_ids.numel()

print("Total Token Positions:", total_positions)


Batch Input IDs Shape: torch.Size([16, 5, 128])
Total Token Positions: 10240


## Question - 5

Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.
Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [11]:
# Multiple-Choice Model Outputs

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice


# Label Encoding
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
train_df["label"] = train_df["answer"].map(label_map)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")


# Prepare row index 0
row = train_df.iloc[0]

prompts = [str(row["prompt"])] * 5
choices = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

encoding = tokenizer(
    prompts,
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Add batch dimension
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)


# Q5: Forward pass without labels
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Logits Shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits)
print("Number of logits:", outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits Shape: torch.Size([1, 5])
Logits:
tensor([[0.3193, 0.3572, 0.3690, 0.3595, 0.3501]])
Number of logits: 5


## Question - 6

Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [12]:
# Q6: Forward pass with labels

labels = torch.tensor([row["label"]])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print("\nLoss:")
print(outputs.loss)

print("Loss Tensor Shape:", outputs.loss.shape)
print("Number of Dimensions:", outputs.loss.dim())


Loss:
tensor(1.6034, grad_fn=<NllLossBackward0>)
Loss Tensor Shape: torch.Size([])
Number of Dimensions: 0


## Question - 7
**LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.**


Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [13]:
# Q7. LoRA Trainable Parameters

import pandas as pd
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

# Load base model
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# LoRA Configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable Parameters:", trainable_params)

# Optional
model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable Parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


## Question - 8

**Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.**

Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [14]:
# Q8. Hugging Face Dataset Preparation

import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer


# Label Encoding
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
train_df["label"] = train_df["answer"].map(label_map)

# Keep first 100 rows
train_df = train_df.iloc[:100].reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

data = []

for _, row in train_df.iterrows():

    prompts = [str(row["prompt"])] * 5

    choices = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    enc = tokenizer(
        prompts,
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    data.append({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(row["label"])
    })

dataset = Dataset.from_list(data)

print(dataset)

print("\nFirst item:")
print(dataset[0])

print("\ninput_ids shape:")
print(len(dataset[0]["input_ids"]), "x", len(dataset[0]["input_ids"][0]))

print("\nNumber of tokenized choices:")
print(len(dataset[0]["input_ids"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

First item:
{'input_ids': [[101, 4060, 1996, 2190, 2825, 3437, 1024, 2054, 2003, 3235, 2002, 5178, 13327, 1005, 1055, 3193, 2006, 1996, 3276, 2090, 2051, 1998, 2529, 4598, 1029, 2426, 1996, 3205, 7047, 1012, 102, 3235, 2002, 5178, 13327, 7164, 2008, 4286, 4839, 2306, 1037, 2051, 22961, 2008, 2003, 10709, 1998, 2515, 2025, 2031, 1037, 4225, 2927, 2030, 2203, 1012, 1996, 3276, 2000, 1996, 2627, 7336, 21894, 2009, 2004, 1037, 3439, 3690, 1010, 1998, 1996, 3276, 2000, 1996, 2925, 7336, 4526, 1037, 2088, 2008, 2097, 18094, 3458, 2028, 1005, 1055, 2219, 2051, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 4060, 1996, 2190, 2825, 3437, 1024, 2054, 2003, 3235, 2002, 5178, 13327, 1005, 1055, 3193, 2006, 1996, 3276, 2090, 2051, 1998, 2529, 4598, 1029, 2426, 1996, 3205, 7047, 1012, 102, 3235, 2002, 5178, 13327, 7164, 2008, 42

## Question - 9

***Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.***

Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [15]:
# Q9 Tiny LoRA Fine-Tuning

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType


label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train_df["label"] = train_df["answer"].map(label_map)

train_df = train_df.iloc[:32].reset_index(drop=True)


# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

examples = []

for _, row in train_df.iterrows():

    prompts = [str(row["prompt"])] * 5

    choices = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    enc = tokenizer(
        prompts,
        choices,
        truncation=True,
        padding="max_length",
        max_length=64
    )

    examples.append({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(row["label"])
    })

dataset = Dataset.from_list(examples)


# Model + LoRA
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)


# Training Arguments
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

train_result = trainer.train()

print("\nFinal Global Step:")
print(trainer.state.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.295811
2,3.206804
3,3.156040
4,3.151013



Final Global Step:
4


## Question - 10

Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [18]:
# Q10 Inference

row = train_df.iloc[0]

prompts = [str(row["prompt"])] * 5
choices = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

enc = tokenizer(
    prompts,
    choices,
    truncation=True,
    padding="max_length",
    max_length=64,
    return_tensors="pt"
)


device = model.device

input_ids = enc["input_ids"].unsqueeze(0).to(device)
attention_mask = enc["attention_mask"].unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probs = torch.softmax(outputs.logits, dim=1)

print("\nProbabilities:")
print(probs)

print("\nProbability of Option E:")
print(round(probs[0,4].item(),4))


Probabilities:
tensor([[0.2035, 0.1962, 0.2020, 0.1995, 0.1989]], device='cuda:0')

Probability of Option E:
0.1989
